In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("21-aqe-performance")
dfs = register_views(spark)
emp  = dfs["employees"]
sal  = dfs["salary_history"]
dept = dfs["departments"]
pa   = dfs["project_assignments"]

# ── Section 1 ── AQE overview and config
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("AQE skewJoin enabled:", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
# AQE = Adaptive Query Execution: optimizes query plan at RUNTIME based on actual statistics
# Three main features:
#   1) coalesce partitions — merges small post-shuffle partitions automatically
#   2) skew join handling — splits oversized partitions to balance work
#   3) switch join strategies — can promote SortMergeJoin → BroadcastHashJoin at runtime


# ── Section 2 ── AQE: dynamic partition coalescing
# Without AQE: shuffle.partitions=200 → 200 output partitions even for tiny data
# With AQE: Spark merges small partitions after shuffle
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "20")
df_no_aqe = emp.groupBy("dept_id").agg(F.avg("salary")).cache()
df_no_aqe.count()
print("Without AQE, partitions:", df_no_aqe.rdd.getNumPartitions())

spark.conf.set("spark.sql.adaptive.enabled", "true")
df_aqe = emp.groupBy("dept_id").agg(F.avg("salary")).cache()
df_aqe.count()
print("With AQE, coalesced partitions:", df_aqe.rdd.getNumPartitions())
# AQE merges the 8 non-empty partitions (one per dept) into fewer large ones


# ── Section 3 ── AQE: skew join handling
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
# With AQE, Spark detects skewed partitions (Engineering has 14 rows) and splits them automatically
# Show the plan difference
spark.conf.set("spark.sql.adaptive.enabled", "false")
emp.join(sal, "emp_id").explain()   # no AQE optimization
spark.conf.set("spark.sql.adaptive.enabled", "true")
emp.join(sal, "emp_id").explain()   # AQE may add OptimizeSkewedJoin


# ── Section 4 ── AQE: dynamic broadcast join switching
# AQE can switch a planned SortMergeJoin to BroadcastHashJoin at runtime if
# it discovers one side is actually small after filtering
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1")   # force no auto-broadcast
spark.conf.set("spark.sql.adaptive.enabled", "true")
# AQE may override and use broadcast if the filtered size is small enough
emp.join(dept, "dept_id").explain()   # check for AdaptivePlan in output


# ── Section 5 ── Tuning shuffle.partitions for AQE
# Without AQE: set shuffle.partitions = 2-3x total executor cores
# With AQE: set it HIGH (e.g., 200) and let AQE coalesce down
# Show the three settings and their effect
spark.conf.set("spark.sql.adaptive.enabled", "true")
for setting in ["4", "20", "200"]:
    spark.conf.set("spark.sql.shuffle.partitions", setting)
    df = emp.groupBy("dept_id").agg(F.sum("salary"))
    df.count()   # trigger
    print(f"shuffle.partitions={setting}, result partitions={df.rdd.getNumPartitions()}")


# ── Section 6 ── AQE + cost-based optimizer (CBO)
# CBO uses table statistics to pick join strategy before execution
# AQE refines at runtime
# Collect stats (normally happens on managed tables; on in-memory DFs this is a no-op)
spark.conf.set("spark.sql.cbo.enabled", "true")
spark.conf.set("spark.sql.statistics.histogram.enabled", "true")
# For in-memory DataFrames, show the explain output only
emp.join(sal, "emp_id").join(dept, emp.dept_id == dept.dept_id, "left").explain(extended=True)


# ── Section 7 ── Key AQE config reference
configs = {
    "spark.sql.adaptive.enabled":                                 "true",
    "spark.sql.adaptive.skewJoin.enabled":                        "true",
    "spark.sql.adaptive.coalescePartitions.enabled":              "true",
    "spark.sql.adaptive.coalescePartitions.minPartitionNum":      "1",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes":            "64m",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor":          "5",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes":"256m",
}
for k, v in configs.items():
    print(f"  {k} = {v}")


stop_and_wait(spark)